In [1]:
import numpy as np
import time
from gridcp import GridDetector
from gridcp.calibration import (
    calibrate_threshold,
    draw_samples,
    mc_alarm_times,
    with_calibrated_threshold,
)
from gridcp.scores import MeanCUSUM, MeanCUSUMUnknownVariance

## New API calibration + run (old sandbox equivalent)
This section mirrors the old API sandbox flow for univariate mean-change detection, but uses the new API and `gridcp.calibration` helpers.

In [2]:
stream_len = 500
n_paths_calibrate = 1000
n_paths_eval = 1000
changepoint = stream_len // 2

### Known variance univariate mean change

In [3]:
def null_sampler(rng):
    return rng.normal(loc=0.0, scale=1.0)


def pre_change_sampler(rng):
    return rng.normal(loc=0.0, scale=1.0)


def post_change_sampler(rng, loc=0.0):
    return rng.normal(loc=loc, scale=1.0)


rng_cal = np.random.default_rng(42)
rng_data = np.random.default_rng(123)

In [4]:
# Known variance: calibrate threshold under null ---- NONPARALLELL
known_score = MeanCUSUM(n_features=1)
critical_value_known_var = calibrate_threshold(
    known_score,
    alpha=0.05,
    stream_len=stream_len,
    n_paths=n_paths_calibrate,
    pre_sampler=null_sampler,
    rng=rng_cal,
    n_features=1,
    parallel=False,
)
known_detector = GridDetector(score=known_score, threshold=critical_value_known_var)
print("Known-variance critical value:", critical_value_known_var)

Known-variance critical value: 3.0597858330282586


In [5]:
# Known variance: calibrate threshold under null ---- Parallell
known_score = MeanCUSUM(n_features=1)
critical_value_known_var = calibrate_threshold(
    known_score,
    alpha=0.05,
    stream_len=stream_len,
    n_paths=n_paths_calibrate,
    pre_sampler=null_sampler,
    rng=rng_cal,
    n_features=1,
    parallel=True,
    n_cores=8,
    strict_equivalence=False,
)
known_detector = GridDetector(score=known_score, threshold=critical_value_known_var)
print("Known-variance critical value:", critical_value_known_var)

Known-variance critical value: 2.9386858175272534


In [10]:
# Simulate streams and record first alarm time in each path.
detect_times = mc_alarm_times(
    detector=known_detector,
    n_paths=n_paths_eval,
    stream_len=stream_len,
    n_features=1,
    pre_sampler=pre_change_sampler,
    post_sampler=post_change_sampler,
    changepoint=changepoint,
    rng=rng_data,
    post_kwargs={"loc": 0.0},
)

print("Known-variance false alarm prob:", np.mean(detect_times <= stream_len))

Known-variance false alarm prob: 0.061


In [5]:
import gridcp.old_api as old_api

detector = old_api.make_univariate_mean_change_detector()
start = time.perf_counter()
detector.calibrate_false_alarm(
    alpha=0.05,
    N=stream_len,
    K=n_paths_calibrate,
    null_dist=np.random.normal,
)
stop = time.perf_counter()
print(detector._state["penalty_constant"])

2.939190545771412


In [7]:
# Unknown variance: calibrate threshold under null.
unknown_score = MeanCUSUMUnknownVariance(n_features=1)
critical_value_unknown_var = calibrate_threshold(
    unknown_score,
    alpha=0.05,
    stream_len=stream_len,
    n_paths=n_paths_calibrate,
    pre_sampler=null_sampler,
    rng=np.random.default_rng(84),
    n_features=1,
)
unknown_detector = with_calibrated_threshold(
    GridDetector(score=unknown_score, threshold=1.0),
    critical_value_unknown_var,
)
print("Unknown-variance critical value:", critical_value_unknown_var)

# Reuse the same changed sample paths to estimate detection delay.
detect_times_unknown = np.ones(n_paths_eval, dtype=np.int64) * stream_len
for path_idx in range(n_paths_eval):
    state = unknown_detector.init_state()
    for t in range(stream_len):
        state, out = unknown_detector.update(state, paths[path_idx, t])
        if out["alarm"]:
            detect_times_unknown[path_idx] = t + 1
            break

print(
    "Unknown-variance avg detection delay:",
    np.mean(detect_times_unknown) - changepoint,
)

Unknown-variance critical value: 2.5327244321434366
Unknown-variance avg detection delay: 228.079
